# INF2005 steganography demonstration

This notebook uses the implementation in the `stego` package. Run it from the repository root.

In [ ]:
import sys
import tempfile
import wave
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "stego" / "__init__.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from IPython.display import display
from PIL import Image

from stego import *

_demo_temp = tempfile.TemporaryDirectory(prefix="inf2005-demo-")
DEMO_DIR = Path(_demo_temp.name)
print(f"Temporary demonstration files: {DEMO_DIR}")

## 1. Setup

Generate one signing keypair and save the public key for verification.

In [ ]:
private_key, public_key = generate_rsa_keypair()
public_key_path = DEMO_DIR / "demo-public-key.pem"
save_rsa_public_key_pem(public_key, public_key_path)
verification_key = load_rsa_public_key_pem(public_key_path)
print(f"Public key: {public_key_path}")
print(display_rsa_public_key_fingerprint(verification_key))

## 2. Encode a PNG

Use three LSBs and a non-zero start. The reported ratio says how much of the cover remains represented in the media hash.

In [ ]:
height, width = 160, 160
yy, xx = np.mgrid[:height, :width]
cover_image = np.stack(((xx * 3 + yy * 0) % 256, (yy * 3 + xx * 0) % 256, ((xx + yy) * 2) % 256), axis=2).astype(np.uint8)
cover_png = DEMO_DIR / "cover.png"
stego_png = DEMO_DIR / "stego-k3.png"
Image.fromarray(cover_image, mode="RGB").save(cover_png)
png_layout, png_payload = encode_png(
    cover_png, stego_png, private_key, start_unit=97, lsb_count=3,
    user_payload=b"INF2005 image demonstration", metadata=b"kind=png",
)
png_preserved = preserved_bit_count(png_layout.total_units, png_layout.footprint, png_layout.lsb_count)
png_ratio = png_preserved / (png_layout.total_units * 8)
print({
    "payload_length": png_layout.payload_length,
    "footprint": png_layout.footprint,
    "pad_bits": png_layout.pad_bits,
    "preserved_bits": png_preserved,
    "preserved_ratio": f"{png_ratio:.2%}",
})
png_comparison = Image.new("RGB", (width * 2, height))
png_comparison.paste(Image.open(cover_png), (0, 0))
png_comparison.paste(Image.open(stego_png), (width, 0))
print("Cover (left) and stego image (right):")
display(png_comparison)

## 3. Verify the PNG

In [ ]:
png_result = verify_png(stego_png, verification_key)
print("Verdict:", png_result.verdict)
print("Media ID:", png_result.payload.media_id)
print("Timestamp:", png_result.payload.timestamp)
print("Nonce:", png_result.payload.nonce.hex())
print("User payload:", png_result.payload.user_payload)
print("Metadata:", png_result.payload.metadata)

## 4. Change the image

Change one carrier byte outside the embedding footprint. The signed packet remains readable, but the media hash reports tampering.

In [ ]:
tampered_png = DEMO_DIR / "tampered.png"
stego_array = load_png_from_path(stego_png)
stego_carrier = rgb_array_to_carrier(stego_array)
outside_index = png_layout.start_unit + png_layout.footprint
stego_carrier[outside_index] ^= np.uint8(1)
save_rgb_png_to_path(carrier_to_rgb_array(stego_carrier, stego_array.shape), tampered_png)
print("Verdict after image change:", verify_png(tampered_png, verification_key).verdict)

## 5. Encode, verify, and change a WAV

In [ ]:
cover_wav = DEMO_DIR / "cover.wav"
stego_wav = DEMO_DIR / "stego.wav"
tampered_wav = DEMO_DIR / "tampered.wav"
samples = (128 + 90 * np.sin(2 * np.pi * 440 * np.arange(32000) / 8000)).astype(np.uint8).tobytes()
with wave.open(str(cover_wav), "wb") as wav_file:
    wav_file.setnchannels(1)
    wav_file.setsampwidth(1)
    wav_file.setframerate(8000)
    wav_file.writeframes(samples)
wav_layout, wav_payload = encode_wav(
    cover_wav, stego_wav, private_key, start_unit=61, lsb_count=5,
    user_payload=b"INF2005 audio demonstration", metadata=b"kind=wav",
)
wav_result = verify_wav(stego_wav, verification_key)
wav_preserved = preserved_bit_count(wav_layout.total_units, wav_layout.footprint, wav_layout.lsb_count)
print("WAV layout:", {"payload_length": wav_layout.payload_length, "footprint": wav_layout.footprint, "pad_bits": wav_layout.pad_bits, "preserved_bits": wav_preserved, "preserved_ratio": f"{wav_preserved / (wav_layout.total_units * 8):.2%}"})
print("WAV verdict:", wav_result.verdict, "payload:", wav_result.payload.user_payload)
wav_data = load_pcm_wav_from_path(stego_wav)
wav_carrier = wav_frame_bytes_to_carrier(wav_data.frame_bytes)
wav_carrier[wav_layout.start_unit + wav_layout.footprint] ^= np.uint8(1)
save_pcm_wav_to_path(wav_data_with_carrier(wav_data, wav_carrier), tampered_wav)
print("WAV verdict after frame change:", verify_wav(tampered_wav, verification_key).verdict)

## 6. Use the wrong public key

In [ ]:
_, unrelated_public_key = generate_rsa_keypair()
print("Wrong-key verdict:", verify_png(stego_png, unrelated_public_key).verdict)

## 7. Compare k=1 and k=8

A larger `k` uses fewer carrier units, but changes more bits in each unit inside the footprint.

In [ ]:
comparison = []
comparison_images = []
for k in (1, 8):
    output = DEMO_DIR / f"stego-k{k}.png"
    layout, _ = encode_png(cover_png, output, private_key, 0, k, b"same payload", b"comparison")
    result = verify_png(output, verification_key)
    saved = np.array(Image.open(output), dtype=np.uint8)
    changed_values = np.count_nonzero(saved != cover_image)
    preserved = preserved_bit_count(layout.total_units, layout.footprint, layout.lsb_count)
    packet_bits = (PACKET_HEADER_SIZE + layout.payload_length + RSA_SIGNATURE_SIZE) * 8
    comparison.append({"k": k, "capacity_bits": layout.total_units * k, "packet_bits": packet_bits, "footprint": layout.footprint, "changed_values": int(changed_values), "preserved_ratio": f"{preserved / (layout.total_units * 8):.2%}", "verdict": result.verdict})
    comparison_images.append(saved)
for row in comparison: print(row)
k_comparison = Image.new("RGB", (width * 3, height))
for index, image in enumerate([cover_image, *comparison_images]):
    k_comparison.paste(Image.fromarray(image, mode="RGB"), (index * width, 0))
print("Cover (left), k=1 (middle), and k=8 (right):")
display(k_comparison)

## Finish

Remove the temporary demonstration files.

In [ ]:
_demo_temp.cleanup()
print("Temporary demonstration files removed.")